In [ ]:
%python
from pyspark.sql.functions import when, col, regexp_replace, trim

# AWS credentials are read from environment variables / Databricks secrets.
# Never hardcode credentials in code.
import os

spark.conf.set("fs.s3a.access.key", os.environ["AWS_ACCESS_KEY_ID"])
spark.conf.set("fs.s3a.secret.key", os.environ["AWS_SECRET_ACCESS_KEY"])
spark.conf.set("fs.s3a.endpoint", "s3.amazonaws.com")


In [ ]:
%run ./audit_utils

In [ ]:
def create_tables(filepath,dbname,table_name):
    df = spark.read.format("parquet").load(filepath)
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {dbname}")
    new_column_names = [col.replace(' ', '_').replace('-', '_').replace('(', '').replace(')','').replace('.','').replace('___','_') for col in df.columns]
    df = df.toDF(*new_column_names)
    check_and_store_md5_result = check_and_store_md5(df, "Bronze", "EV_Data", schema_flag ="N")
    if check_and_store_md5_result == True:
        df.write.format("delta").mode("append").saveAsTable(f"{dbname}.{table_name}")
    if check_and_store_md5_result == False:
        df.write.option("mergeSchema", "true").format("delta").mode("append").saveAsTable(f"{dbname}.{table_name}")

In [ ]:
def create_meta_tables(filepath,dbname,table_name):
    df = spark.read.format("parquet").load(filepath)
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {dbname}")
    new_column_names = [col.replace(' ', '_').replace('-', '_').replace('(', '').replace(')','').replace('.','').replace('___','_') for col in df.columns]
    df = df.toDF(*new_column_names)
    df.write.option("mergeSchema", "true").format("delta").mode("overwrite").saveAsTable(f"{dbname}.{table_name}")


In [ ]:
veh_data_path = "s3a://evdata-test/derived/vehicle_data/"
dbname = "bronze"
table_name = "vehicle_data"

# Create tables using function
update_audit_table("EV_Data", progress="Ingest_Bronze", status="In_Progress", start=True)
create_tables(veh_data_path,dbname,table_name)
display(spark.sql(f"SELECT * FROM {dbname}.{table_name} LIMIT 10"))

In [ ]:
update_audit_table("EV_Data", progress="Ingest_Bronze", status="Completed", end=True)

In [ ]:
col_meta_path = "s3a://evdata-test/derived/columns_metadata/"
table_name = "columns_metadata"
create_meta_tables(col_meta_path,dbname,table_name)
display(spark.sql(f"SELECT * FROM {dbname}.{table_name} LIMIT 10"))

In [ ]:
tab_meta_path = "s3a://evdata-test/derived/table_metadata/"
table_name = "table_metadata"
create_meta_tables(tab_meta_path,dbname,table_name)
display(spark.sql(f"SELECT * FROM {dbname}.{table_name} LIMIT 10"))